<div style="display: flex; align-items: center;">
  <img src="../Images/Logos/DecisionIntelligence-DecisionFraming-Logo.png" width="60px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Decision Framing - Six Thinking Hats</span>
</div>

Decision Intelligence applied in this module:

* Using decision frames to evaluate a major purchase from multiple perspectives.
* Applying Edward de Bono's Six Thinking Hats as reusable decision lenses.
* Asking GenAI to prioritize the most useful frames for a high-stakes decision.
* Combining prompt design and Microsoft.Extensions.AI to produce strict Markdown-table outputs.

### Step 1 - Initialize Configuration Builder & Build the AI Extensions Responses API Orchestration

In [1]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.8"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.8"
#r "nuget: System.Text.Json, 10.0.8"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.8 Microsoft.Extensions.Configuration.Json, 10.0.8 System.Text.Json, 10.0.8

In [2]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.6.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Responses;
using System.ClientModel;
using System.Collections.Generic;
using System.Threading.Tasks;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI.
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service.
IChatClient chatClient;

#pragma warning disable OPENAI001

if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    // Get the ResponsesClient from AzureOpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(openAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    // Get the ResponsesClient from OpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

var decisionFrameSystemPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems,
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, balanced, analytical, and pragmatic.
Improve the user's judgment rather than simply making the choice for them.

Output Format Instructions:
Return only a valid Markdown table.
Do not include prose before or after the table.
Do not wrap the table in triple backticks.
Do not use Markdown headings anywhere in the response.
Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
Do not use horizontal rules of any kind.
Do not use ---, ***, or ___ anywhere in the response.
Do not use bullet lists or numbered lists inside table cells.
Use plain text column headers only.
If line breaks are needed inside a cell, use <br>.
Escape any literal pipe characters inside cell content so the table structure is not broken.
Make the table syntactically valid Markdown with exactly one header row and one separator row.
""";

async Task<string> RunDecisionFramePromptAsync(string userPrompt, ChatOptions? options = null)
{
    var chatMessages = new List<ChatMessage>
    {
        new(ChatRole.System, decisionFrameSystemPrompt),
        new(ChatRole.User, userPrompt)
    };

    ChatResponse response = await chatClient.GetResponseAsync(chatMessages, options);
    return response.Text ?? string.Empty;
}

#pragma warning disable OPENAI001

var decisionFrameChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

#pragma warning restore OPENAI001

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.6.0 Microsoft.Extensions.AI.Abstractions, 10.6.0 Microsoft.Extensions.AI.OpenAI, 10.6.0 OpenAI, 2.10.0

Using Azure OpenAI Service



(69,78): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Azure.Core, Version=1.51.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'Azure.Core, Version=1.53.0.0, Culture=neutral, PublicKeyToken=92742159

### Step 2 - Decision Frames (Thinking Hats) Scenario for a Major Purchase

> 📜 **_"The mere formulation of a problem is far more essential than its solution, which may be merely a matter of mathematical or experimental skill. To raise new questions, new possibilities, to regard old problems from a new angle requires creative imagination and marks real advances in science."_**
>
> -- <cite>Albert Einstein (Theoretical Physicist, Best known for developing the theory of relativity)</cite>  

The code below lists potential decision frames using the Six Thinking Hats framework for purchasing a house. Generative AI can help apply each hat as a distinct decision lens, making it easier to inspect the same choice from multiple useful perspectives.

In [3]:
// Decision Frames Prompt that lists the Six Thinking Hats approach to purchasing a house.
var decisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
Create one row for each of the six thinking hats.
For each row, explain how that decision frame should guide the house-purchase decision.
Include practical questions and evidence the buyer should gather.

Output Requirements:
Return one Markdown table with these columns:
Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value
""";

var decisionFramesResponseText = await RunDecisionFramePromptAsync(
    decisionFramesPrompt,
    decisionFrameChatOptions);

decisionFramesResponseText.DisplayAs("text/markdown");

| Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value |
|---|---|---|---|---|
| White Hat | Facts, data, and known information | What is the asking price, recent sale price, monthly payment estimate, property tax, insurance, HOA, utilities, commute time, school ratings, and maintenance cost? What are the objective strengths and weaknesses of the property and neighborhood? | Comparative market analysis, mortgage pre-approval, tax records, inspection report, utility bills, HOA documents, flood and zoning data, commute estimates, school and crime data | Grounds the decision in verifiable facts and reduces guesswork, helping you compare homes consistently |
| Red Hat | Feelings, intuition, and emotional response | How do I feel when I walk through the home and neighborhood? Do I feel calm, excited, constrained, or uneasy? Does the space fit my lifestyle and long-term sense of “home”? | Personal reactions after each visit, feedback from all decision-makers, notes on what felt energizing or off-putting, impressions from different times of day | Surfaces emotional fit and hidden concerns that data alone may miss, especially for a high-stakes, long-term purchase |
| Black Hat | Risks, downsides, and failure modes | What could go wrong financially, structurally, or socially? Could rates rise, repairs exceed budget, resale be weak, or the area change negatively? What assumptions are fragile? | Inspection findings, repair estimates, reserve and emergency fund analysis, stress test of monthly budget, neighborhood trend data, legal/title review, insurance exclusions | Protects against optimism bias and helps avoid costly mistakes by stress-testing the purchase |
| Yellow Hat | Benefits, value, and upside | What are the best-case and likely benefits of buying this home? Will it improve stability, commute, quality of life, equity building, or flexibility? Is there renovation upside or resale potential? | Appreciation trends, rent-vs-buy comparison, renovation ROI estimates, mortgage amortization schedule, lifestyle benefits by location, long-term neighborhood plans | Highlights the strategic upside and clarifies whether the home meaningfully supports your goals |
| Green Hat | Creativity, alternatives, and options | Could I negotiate price, ask for repairs, buy with a contingency, or wait for another listing? Is there a different neighborhood, property type, or renovation plan that better fits my needs? | Alternative listings, negotiation benchmarks, renovation scenarios, lender options, closing-cost comparisons, timeline flexibility, “must-have” versus “nice-to-have” list | Expands the solution space and helps you avoid false dilemmas, often revealing better or lower-risk paths |
| Blue Hat | Process, structure, and decision discipline | What is the decision criteria, timeline, and threshold for saying yes or no? Have I collected enough evidence and involved the right people? What is the review process if new information appears? | Decision checklist, weighted criteria matrix, decision deadline, advisor input, pre-commitment rules, documentation of tradeoffs and non-negotiables | Keeps the process organized, prevents emotional drift, and ensures the final decision is deliberate and repeatable |

This can be enhanced further by asking GenAI to suggest which decision frames might be most useful for a high-stakes decision. The prompt below lists all of the decision frames, prioritizes them by expected impact and influence, and recommends which frames deserve the most attention.

This lets the AI automation help optimize the decision process while the human decision maker remains responsible for the actual decision selection and execution.

In [4]:
// Decision Frames Prompt that ranks the Six Thinking Hats approach to purchasing a house.
var decisionFramesListPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
List the six thinking hats in the order of expected impact and influence from most to least for a house-purchase decision.
For each thinking hat, explain why it has that ranking.
Recommend whether the buyer should focus on that hat heavily, moderately, or lightly.

Output Requirements:
Return one Markdown table with these columns:
Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus
""";

var decisionFramesListResponseText = await RunDecisionFramePromptAsync(
    decisionFramesListPrompt,
    decisionFrameChatOptions);

decisionFramesListResponseText.DisplayAs("text/markdown");

| Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus |
|---|---|---|---|---|
| 1 | White Hat | Buying a home is a major financial commitment, so objective facts usually drive the quality of the decision: budget, interest rate, monthly payment, taxes, insurance, comps, inspection results, commute time, school data, and resale fundamentals. If the facts are wrong, every other judgment is weakened. | Highest | Heavily |
| 2 | Black Hat | The downside risk is large and often expensive: hidden defects, overpaying, market softness, repair surprises, HOA constraints, title issues, and affordability strain. This hat protects the buyer from avoidable regret and financial stress. | High | Heavily |
| 3 | Yellow Hat | A house is not only a liability; it can also deliver long-term value through stability, equity building, lifestyle fit, tax advantages, and future flexibility. This hat helps balance the caution of Black Hat with a realistic view of upside. | Medium-High | Moderately |
| 4 | Blue Hat | A disciplined process prevents emotional drift: setting criteria, defining a maximum budget, comparing options consistently, deciding when to walk away, and sequencing due diligence. For a house purchase, good process often matters because it prevents costly mistakes. | Medium-High | Moderately |
| 5 | Red Hat | Emotions matter because a home is also a lived-in environment, not just an asset. Comfort, stress, sense of belonging, and family reactions can reveal fit that data alone misses, but this hat should not override affordability or risk analysis. | Medium | Lightly |
| 6 | Green Hat | Creativity helps with alternative neighborhoods, renovation potential, financing structures, negotiation tactics, or rethinking what “ideal” looks like. Useful, but for most home purchases it has less influence than facts, risk control, and process discipline. | Medium-Low | Lightly |

So far, GenAI has helped create decision frames with the Six Thinking Hats framework, rank them from most to least impactful, and identify where to focus attention. The next step is to apply the selected decision frames directly to the house-purchase decision.

This notebook uses a single optimized prompt for that final step: the model performs the ranking and frame selection internally, then returns only the applied decision-frame approach as a Markdown table.

#### Single Decision Prompt

In [5]:
// Decision Frames Prompt that internally ranks and selects the strongest frames, then only returns the applied approach.
var optimizedDecisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Internal Instructions:
Internally identify an approach for each of the six thinking hats.
Internally rank the thinking hats by expected impact and influence for the house-purchase decision.
Internally select the most useful decision frames to focus on.
Do not show the internal ranking, hidden selection notes, or hidden reasoning steps.

Visible Output Instructions:
From the internally selected decision frames, create an applied approach for purchasing a house.
The response should help the buyer know what to inspect, what questions to ask, what evidence to gather, and what action to take.

Output Requirements:
Return one Markdown table with these columns:
Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action
""";

var optimizedDecisionFramesResponseText = await RunDecisionFramePromptAsync(
    optimizedDecisionFramesPrompt,
    decisionFrameChatOptions);

optimizedDecisionFramesResponseText.DisplayAs("text/markdown");

| Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action |
|---|---|---|---|---|
| White Hat: Facts and data | Build a fact base before forming opinions. Treat the home as an information problem first. Inspect the property, documents, and market context. | What are the exact square footage, lot size, age, permits, taxes, HOA rules, utility costs, and repair history? What are comparable recent sale prices? What is the inspection history? | Seller disclosure, listing details, county records, comparable sales, inspection reports, appraisal, permit history, HOA documents, flood and hazard maps, utility bills. | Collect and organize all documents in one place. Verify any missing or inconsistent data before making an offer. |
| Red Hat: Gut feeling and fit | Notice emotional reactions and instinctive comfort. A house must feel livable, not just look good on paper. | Do I feel calm, energized, or uneasy in this space? Does the neighborhood feel safe and welcoming? Can I imagine daily life here? | Immediate emotional reaction during visits, sense of noise, light, layout flow, privacy, and neighborhood vibe. Reactions from all decision makers after separate visits. | Record first impressions immediately after each showing. Compare gut reactions across times of day and with different weather or traffic conditions. |
| Black Hat: Risks and downside | Stress-test the decision for failure modes. Focus on hidden costs, defects, and worst-case scenarios. | What could make this a bad purchase? What repairs are likely soon? Could the area decline, flood, or become hard to resell? Can we comfortably afford the mortgage if rates, taxes, or income change? | Structural inspection findings, roof and HVAC age, mold or pest evidence, insurance quotes, flood or wildfire risk, resale comparables, debt-to-income ratio, cash reserve needs. | Add protective contingencies, price for repairs, cap your maximum offer based on risk, and walk away if major red flags are unresolved. |
| Yellow Hat: Benefits and value | Evaluate the upside in practical terms. Focus on long-term value, lifestyle gains, and strategic advantages. | How does this home improve daily life, commute, family needs, or remote work? Is there likely appreciation, strong rental potential, or tax benefit? Does the layout support future needs? | School quality, commute time, neighborhood demand, historical price trends, renovation potential, energy efficiency, and livability features such as bedrooms, yard, or office space. | Prioritize homes that create durable value, not just short-term appeal. If benefits are strong, move quickly with a competitive but disciplined offer. |
| Green Hat: Alternatives and creativity | Expand the option set before committing. Look for ways to solve problems creatively instead of accepting the first apparent choice. | Is there another neighborhood, house type, or timeline that fits better? Could renovations, concessions, or financing changes improve the fit? Would renting longer, waiting, or buying smaller be smarter? | Alternative listings, renovation estimates, seller concessions, rate buy-down options, inspection negotiation room, and scenarios for different budgets or locations. | Compare at least three viable alternatives. Consider creative structures such as repair credits, price adjustments, or a delayed closing if they improve the deal. |
| Blue Hat: Process and decision control | Manage the decision process deliberately. Define criteria, sequence, and decision thresholds before emotional momentum takes over. | What are our must-haves, nice-to-haves, deal breakers, budget ceiling, and decision deadline? Who approves what? What evidence is required before offer, contract, and closing? | Written criteria, scorecard, budget model, timeline, contingency plan, lender preapproval, professional inspections, and agreed decision rules. | Use a formal home-buying checklist and scorecard. Set go or no-go thresholds in advance, review them after each visit, and only proceed when the home meets the pre-set standard. |